In [1]:
model_name = "vit-ragdoll"


import iree
import iree.compiler
import iree.runtime
import torch
import numpy as np
from torch import nn
from torchvision import models

import warnings
warnings.filterwarnings("ignore")

from timeit import timeit as ti
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from ragdoll.compiler import *
from ragdoll.benchmark import *
import ragdoll

def get_dataframe(backward, item):
    return pd.concat([
        pd.DataFrame({
            "time": backward,
            "pass": "Backward",
            "item": item
        }, index=[0]),
    ])

def timeit(stmt, n=1):
    return ti(stmt, globals=globals(), number=n) * 1000 / n



BENCHMARK_REPEAT=33
df = pd.DataFrame()

def load_executable(fb_file):
    config = iree.runtime.system_api.Config("cuda")
    vmi = iree.runtime.VmInstance()
    # replace compile with args of fatbin type
    # fb_file = ragdoll.compile(mlir, "gpu", "input", "codegen", benchmark=True)
    with open(fb_file, 'rb') as f:
        binary_data = f.read()
    vmm = iree.runtime.VmModule.from_flatbuffer(vmi, binary_data)
    vmo = iree.runtime.load_vm_module(vmm, config)
    return vmo

import torch
from torch import nn
from torchvision import models
import pandas as pd

MAGIC_NUM = 7777e-5

device = torch.device("cuda:0")
model = models.vit_b_16().train(False)
model.load_state_dict({k: torch.ones_like(v) * MAGIC_NUM for k, v in model.state_dict().items()})
model = model.to(device)

df = pd.DataFrame()

!cpupower frequency-set --governor performance

Setting cpu: 0
Setting cpu: 1
Setting cpu: 2
Setting cpu: 3
Setting cpu: 4
Setting cpu: 5
Setting cpu: 6
Setting cpu: 7
Setting cpu: 8
Setting cpu: 9
Setting cpu: 10
Setting cpu: 11
Setting cpu: 12
Setting cpu: 13
Setting cpu: 14
Setting cpu: 15
Setting cpu: 16
Setting cpu: 17
Setting cpu: 18
Setting cpu: 19
Setting cpu: 20
Setting cpu: 21
Setting cpu: 22
Setting cpu: 23
Setting cpu: 24
Setting cpu: 25
Setting cpu: 26
Setting cpu: 27
Setting cpu: 28
Setting cpu: 29
Setting cpu: 30
Setting cpu: 31
Setting cpu: 32
Setting cpu: 33
Setting cpu: 34
Setting cpu: 35
Setting cpu: 36
Setting cpu: 37
Setting cpu: 38
Setting cpu: 39
Setting cpu: 40
Setting cpu: 41
Setting cpu: 42
Setting cpu: 43
Setting cpu: 44
Setting cpu: 45
Setting cpu: 46
Setting cpu: 47


In [2]:
import gc
model_file_base = "vit.mlir"
strategy = "heuristic"
ragdoll_results = []
ragdoll_throughput = []
for bs in range(1, 37):
    print("measuring #", bs)
    model_file = model_file_base + ".bs{}".format(bs)
    source_file = model_file + ".{}".format(strategy)
    target_file = source_file + ".vmfb"
    """
    # gen model with specified batch-size
    !ragdoll-opt {model_file_base} --ragdoll-autodiff-prepare-batch-size=batchsize={bs} > {model_file}
    
    !ragdoll-opt {model_file}  \
    --canonicalize \
    --enable-cse-in-legalizer \
    --symbol-dce \
    --ragdoll-autodiff-vjp-public-functions='strategy=heuristic' \
    --ragdoll-autodiff-vjp \
    --inline \
    --ragdoll-autodiff-inline-function-call \
    --ragdoll-initialisation \
    --eliminate-empty-tensors \
    --ragdoll-legalise-to-iree-compatibility \
    --ragdoll-raise-linalg-to-tosa \
    --ragdoll-forward-func-removal \
    --canonicalize \
    --cse > {source_file}
    """
    !iree-compile {source_file} \
    -o {target_file} \
    --iree-codegen-llvm-distribution-size=8 \
    --iree-hal-target-backends=cuda \
    --iree-hal-benchmark-dispatch-repeat-count={BENCHMARK_REPEAT} \
    --iree-hal-cuda-llvm-target-arch=sm_70
    
    
    #ragdoll_binary = load_executable(target_file)


    #try:
        #f1 = timeit("ragdoll_binary.forward(image_np_t)") / BENCHMARK_REPEAT
        #print('ragdoll-opt1-gpu-forward in timeit: ', f1)
    !iree-benchmark-module \
    --module={target_file} \
    --device=cuda \
    --function=dforward \
    --input={bs}x1000xf32 \
    --batch_size={BENCHMARK_REPEAT} \
    --benchmark_repetitions=1 \
    --batch_concurrency=1 \
    --benchmark_min_time=0.1s \
    --print_statistics=true
    """
    b1 = ragdoll_model_benchmark(
        target_file,
        "dforward",
        [(bs, 1000)],
        device='gpu',
        warmups=15,
        repetitions=BENCHMARK_REPEAT, 
        measure_count=1,
        record_mem=True,
        verbose=True)
    b1 = np.mean(b1)
    """
    """
    b1 = timeit("ragdoll_binary.dforward(grad_np)") / BENCHMARK_REPEAT


    print(b1)
    
    ragdoll_results.append(b1)
    ragdoll_throughput.append(bs/b1)
    """


measuring # 1
2024-03-29T23:59:29+08:00
Running /root/miniconda3/envs/albert-research-py310/lib/python3.10/site-packages/iree/_runtime_libs/iree-benchmark-module
Run on (48 X 4095.72 MHz CPU s)
CPU Caches:
  L1 Data 32 KiB (x24)
  L1 Instruction 32 KiB (x24)
  L2 Unified 512 KiB (x24)
  L3 Unified 16384 KiB (x8)
Load Average: 0.36, 0.38, 0.37
---------------------------------------------------------------------------------------------
Benchmark                                   Time             CPU   Iterations UserCounters...
---------------------------------------------------------------------------------------------
BM_dforward/process_time/real_time       11.4 ms         11.4 ms           33 items_per_second=88.0074/s
[[ iree_hal_allocator_t memory statistics ]]
  HOST_LOCAL:            0B peak /            0B allocated /            0B freed /            0B live
DEVICE_LOCAL:     22593120B peak /     22589120B allocated /     22589120B freed /            0B live
measuring # 2
2024-

In [3]:
for bs in range(1, 27):
    dyn_model = torch.compile(model, backend="inductor")
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = dyn_model(image)
    grad = torch.randn_like(output)
    
    torch.cuda.reset_peak_memory_stats(device=device)
    before = torch.cuda.memory_allocated(device=device)
    print("measuring #", bs)
    baseline_f = timeit("dyn_model(image.to(device))", 333)
    baseline_b = timeit("torch.autograd.grad(output.to(device), [image.to(device)], grad.to(device), retain_graph=True)", 333)
    #print(baseline_f)
    print(baseline_b)

    # 记录操作后的峰值内存使用情况
    peak_memory = torch.cuda.max_memory_allocated(device=device)
    
    # 显示结果
    print(f"Memory used before operation: {before / (1024**2):.2f} MB")
    print(f"Peak memory usage: {peak_memory / (1024**2):.2f} MB")

measuring # 1
17.8325363258655
Memory used before operation: 451.04 MB
Peak memory usage: 791.51 MB
measuring # 2
28.945180531535243
Memory used before operation: 605.01 MB
Peak memory usage: 943.25 MB
measuring # 3
35.212707392558144
Memory used before operation: 719.98 MB
Peak memory usage: 1103.02 MB
measuring # 4
46.344198429168344
Memory used before operation: 826.79 MB
Peak memory usage: 1309.13 MB
measuring # 5
53.333033908259225
Memory used before operation: 947.80 MB
Peak memory usage: 1553.23 MB
measuring # 6
60.399070681372024
Memory used before operation: 1065.27 MB
Peak memory usage: 1790.61 MB
measuring # 7
67.49788712285034
Memory used before operation: 1184.53 MB
Peak memory usage: 2027.81 MB
measuring # 8
78.59869510796783
Memory used before operation: 1304.90 MB
Peak memory usage: 2268.31 MB
measuring # 9
85.26802584305182
Memory used before operation: 1424.72 MB
Peak memory usage: 2507.43 MB
measuring # 10
93.09336364090264
Memory used before operation: 1571.78 MB
Pe

Process ForkProcess-21:
Process ForkProcess-11:
Process ForkProcess-2:
Process ForkProcess-6:
Process ForkProcess-20:
Process ForkProcess-10:
Process ForkProcess-27:
Process ForkProcess-16:
Process ForkProcess-25:
Process ForkProcess-24:
Process ForkProcess-29:
Process ForkProcess-32:
Process ForkProcess-26:
Process ForkProcess-9:
Process ForkProcess-12:
Process ForkProcess-18:
Process ForkProcess-7:
Process ForkProcess-8:
Process ForkProcess-22:
Process ForkProcess-23:
Process ForkProcess-19:
Process ForkProcess-28:
Process ForkProcess-17:
Process ForkProcess-31:
Process ForkProcess-30:


In [4]:
from timeit import timeit as ti
def timeit(stmt, n=1):
    return ti(stmt, globals=globals(), number=n) * 1000 / n
for bs in range(1, 27):
    image = torch.randn(bs, 3, 224, 224, requires_grad=True)
    image = image.to(device)
    output = model(image)
    grad = torch.randn_like(output)
    torch.cuda.reset_peak_memory_stats(device=device)
    before = torch.cuda.memory_allocated(device=device)
    print("measuring #", bs)
    baseline_f = timeit("model(image)", 30)
    baseline_b = timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3)
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    baseline_b = max(baseline_b, timeit("torch.autograd.grad([output], [image], [grad], retain_graph=True)", 3))
    #print(baseline_f)
    print(baseline_b)
    
    # 记录操作后的峰值内存使用情况
    peak_memory = torch.cuda.max_memory_allocated(device=device)
    
    # 显示结果
    print(f"Memory used before operation: {before / (1024**2):.2f} MB")
    print(f"Peak memory usage: {peak_memory / (1024**2):.2f} MB")

measuring # 1
10.170401073992252
Memory used before operation: 465.32 MB
Peak memory usage: 581.43 MB
measuring # 2
17.855240342517693
Memory used before operation: 613.07 MB
Peak memory usage: 871.56 MB
measuring # 3
24.02530734737714
Memory used before operation: 715.18 MB
Peak memory usage: 1088.18 MB
measuring # 4
30.295588076114655
Memory used before operation: 825.95 MB
Peak memory usage: 1307.37 MB
measuring # 5
38.918999334176384
Memory used before operation: 949.13 MB
Peak memory usage: 1552.47 MB
measuring # 6
45.006548364957176
Memory used before operation: 1065.06 MB
Peak memory usage: 1786.33 MB
measuring # 7
51.23367098470529
Memory used before operation: 1186.83 MB
Peak memory usage: 2028.29 MB
measuring # 8
57.64174275100231
Memory used before operation: 1307.47 MB
Peak memory usage: 2270.92 MB
measuring # 9
69.10967050741117
Memory used before operation: 1436.67 MB
Peak memory usage: 2532.72 MB
measuring # 10
72.4889285241564
Memory used before operation: 1543.18 MB
Pe

In [5]:
!sudo cpupower frequency-set --governor powersave

Setting cpu: 0
Setting cpu: 1
Setting cpu: 2
Setting cpu: 3
Setting cpu: 4
Setting cpu: 5
Setting cpu: 6
Setting cpu: 7
Setting cpu: 8
Setting cpu: 9
Setting cpu: 10
Setting cpu: 11
Setting cpu: 12
Setting cpu: 13
Setting cpu: 14
Setting cpu: 15
Setting cpu: 16
Setting cpu: 17
Setting cpu: 18
Setting cpu: 19
Setting cpu: 20
Setting cpu: 21
Setting cpu: 22
Setting cpu: 23
Setting cpu: 24
Setting cpu: 25
Setting cpu: 26
Setting cpu: 27
Setting cpu: 28
Setting cpu: 29
Setting cpu: 30
Setting cpu: 31
Setting cpu: 32
Setting cpu: 33
Setting cpu: 34
Setting cpu: 35
Setting cpu: 36
Setting cpu: 37
Setting cpu: 38
Setting cpu: 39
Setting cpu: 40
Setting cpu: 41
Setting cpu: 42
Setting cpu: 43
Setting cpu: 44
Setting cpu: 45
Setting cpu: 46
Setting cpu: 47
